# Kaggle Paper-Aligned Full Notebook

Runs the paper-style protocol from the Python modules in this repo:

- train on hand-held/probe split: `audio_visual_dataset_default/`
- validate/test on robot split: `audio_visual_dataset_robo_default/`
- tasks: multiclass and binary contact classification
- modes: audio, video/image, fusion
- output: printed F1/Precision/Recall plus saved checkpoints and JSON results

Upload this notebook together with the `coding/` folder so the imports/scripts are available on Kaggle.


## 1. Setup

Set `DATA_ROOT` to the folder containing `audio_visual_dataset_default/` and `audio_visual_dataset_robo_default/`. On Kaggle, change the first path if your dataset name is different.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd

DATA_CANDIDATES = [
    Path('/kaggle/input/datasets/nguynnguynhehe/audio-video-dataset/dataset'),
    Path('/kaggle/input/audio-video-dataset/dataset'),
    Path('/kaggle/input/contact-data/dataset'),
    Path('/kaggle/input/contact_data/dataset'),
    Path('../dataset'),
    Path('dataset'),
]
DATA_ROOT = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])

OUTPUT_DIR = Path('/kaggle/working/outputs') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDED_FILES = {'coding/__init__.py': '"""Kaggle-friendly audio-visual contact classification package."""\n', 'coding/config.py': 'from dataclasses import dataclass\nfrom pathlib import Path\n\n\nLABELS = ("ambient", "leaf", "trunk", "twig")\nLABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}\nID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}\n\n\n@dataclass\nclass DataConfig:\n    data_root: Path\n    target_sample_rate: int = 16000\n    audio_window_sec: float = 0.8\n    n_mels: int = 128\n    n_fft: int = 1024\n    hop_length: int = 256\n    image_size: int = 224\n    skip_missing_files: bool = True\n    train_crop: str = "energy"\n    eval_crop: str = "energy"\n    normalize_audio_db: bool = True\n    spectral_gate: bool = True\n    spectral_gate_noise_percentile: float = 20.0\n    spectral_gate_strength: float = 1.0\n\n\n@dataclass\nclass TrainConfig:\n    epochs: int = 5\n    batch_size: int = 8\n    lr: float = 1e-4\n    weight_decay: float = 1e-4\n    num_workers: int = 2\n    val_size: float = 0.2\n    seed: int = 42\n    use_amp: bool = True\n    class_weights: bool = True\n\n\n@dataclass\nclass ModelConfig:\n    ast_model_name: str = "MIT/ast-finetuned-audioset-10-10-0.4593"\n    clap_model_name: str = "laion/clap-htsat-unfused"\n    fusion_dim: int = 256\n    fusion_heads: int = 4\n    fusion_layers: int = 2\n    fusion_dropout: float = 0.1\n    freeze_pretrained: bool = False\n    ast_input_source: str = "mel"\n', 'coding/audio.py': 'from __future__ import annotations\n\nimport math\nimport wave\n\nimport torch\nimport torch.nn.functional as F\n\n\nclass AudioPipeline:\n    def __init__(\n        self,\n        target_sample_rate: int = 22000,\n        window_sec: float = 0.8,\n        n_mels: int = 128,\n        n_fft: int = 1024,\n        hop_length: int = 256,\n        normalize_db: bool = True,\n        spectral_gate: bool = True,\n        spectral_gate_noise_percentile: float = 20.0,\n        spectral_gate_strength: float = 1.0,\n    ) -> None:\n        try:\n            import torchaudio\n        except (ImportError, OSError):\n            torchaudio = None\n\n        self.torchaudio = torchaudio\n        self.target_sample_rate = target_sample_rate\n        self.window_samples = int(round(target_sample_rate * window_sec))\n        self.normalize_db = normalize_db\n        self.n_fft = n_fft\n        self.hop_length = hop_length\n        self.n_mels = n_mels\n        self.spectral_gate = spectral_gate\n        self.spectral_gate_noise_percentile = spectral_gate_noise_percentile\n        self.spectral_gate_strength = spectral_gate_strength\n        if torchaudio is not None:\n            self.mel = torchaudio.transforms.MelSpectrogram(\n                sample_rate=target_sample_rate,\n                n_fft=n_fft,\n                hop_length=hop_length,\n                n_mels=n_mels,\n                power=2.0,\n            )\n            self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)\n            self.mel_filter = None\n        else:\n            self.mel = None\n            self.to_db = None\n            self.mel_filter = _mel_filterbank(target_sample_rate, n_fft, n_mels)\n        self._resamplers: dict[int, torch.nn.Module] = {}\n\n    def load_waveform(self, path: str):\n        if self.torchaudio is not None:\n            waveform, sample_rate = self.torchaudio.load(path)\n        else:\n            waveform, sample_rate = _load_wav_stdlib(path)\n        if waveform.shape[0] > 1:\n            waveform = waveform.mean(dim=0, keepdim=True)\n        if sample_rate != self.target_sample_rate:\n            if self.torchaudio is not None and sample_rate not in self._resamplers:\n                self._resamplers[sample_rate] = self.torchaudio.transforms.Resample(\n                    orig_freq=sample_rate,\n                    new_freq=self.target_sample_rate,\n                    lowpass_filter_width=64,\n                    rolloff=0.9475937167399596,\n                    resampling_method="sinc_interp_kaiser",\n                )\n            if self.torchaudio is not None:\n                waveform = self._resamplers[sample_rate](waveform)\n            else:\n                waveform = _resample_scipy(waveform, sample_rate, self.target_sample_rate)\n        return waveform\n\n    def crop_or_pad(self, waveform: torch.Tensor, mode: str = "center") -> torch.Tensor:\n        total = waveform.shape[-1]\n        target = self.window_samples\n        if total < target:\n            return F.pad(waveform, (0, target - total))\n        if total == target:\n            return waveform\n\n        max_start = total - target\n        if mode == "random":\n            start = int(torch.randint(0, max_start + 1, (1,)).item())\n        elif mode == "energy":\n            start = self._energy_start(waveform, target)\n        else:\n            start = max_start // 2\n        return waveform[..., start : start + target]\n\n    def _energy_start(self, waveform: torch.Tensor, target: int) -> int:\n        # For 1s -> 0.8s crops this is cheap and picks the loudest contiguous window.\n        energy = waveform.pow(2).mean(dim=0, keepdim=True).unsqueeze(0)\n        kernel = torch.ones(1, 1, target, device=waveform.device)\n        scores = F.conv1d(energy, kernel)\n        return int(scores.argmax(dim=-1).item())\n\n    def waveform_to_mel(self, waveform: torch.Tensor) -> torch.Tensor:\n        if self.torchaudio is not None:\n            mel = self.mel(waveform)\n            mel = self.to_db(mel)\n        else:\n            window = torch.hann_window(self.n_fft, device=waveform.device)\n            spec = torch.stft(\n                waveform,\n                n_fft=self.n_fft,\n                hop_length=self.hop_length,\n                win_length=self.n_fft,\n                window=window,\n                return_complex=True,\n            ).abs().pow(2.0)\n            mel_filter = self.mel_filter.to(waveform.device)\n            mel = torch.matmul(mel_filter, spec.squeeze(0)).unsqueeze(0)\n            mel = 10.0 * torch.log10(torch.clamp(mel, min=1e-10))\n            mel = torch.clamp(mel, min=mel.max() - 80.0)\n        if self.normalize_db:\n            mel = (mel + 80.0) / 80.0\n            mel = mel.clamp(0.0, 1.0)\n        return mel\n\n    def __call__(self, path: str, crop_mode: str = "center") -> torch.Tensor:\n        waveform = self.load_processed_waveform(path, crop_mode)\n        return self.waveform_to_mel(waveform)\n\n    def load_processed_waveform(self, path: str, crop_mode: str = "center") -> torch.Tensor:\n        waveform = self.load_waveform(path)\n        if self.spectral_gate:\n            waveform = self.apply_spectral_gate(waveform)\n        return self.crop_or_pad(waveform, crop_mode)\n\n    def apply_spectral_gate(self, waveform: torch.Tensor) -> torch.Tensor:\n        if waveform.shape[-1] < self.n_fft:\n            return waveform\n        window = torch.hann_window(self.n_fft, device=waveform.device)\n        spec = torch.stft(\n            waveform,\n            n_fft=self.n_fft,\n            hop_length=self.hop_length,\n            win_length=self.n_fft,\n            window=window,\n            return_complex=True,\n        )\n        magnitude = spec.abs()\n        noise = torch.quantile(\n            magnitude,\n            self.spectral_gate_noise_percentile / 100.0,\n            dim=-1,\n            keepdim=True,\n        )\n        gated_mag = (magnitude - self.spectral_gate_strength * noise).clamp_min(0.0)\n        phase = spec / magnitude.clamp_min(1e-8)\n        gated = gated_mag * phase\n        return torch.istft(\n            gated,\n            n_fft=self.n_fft,\n            hop_length=self.hop_length,\n            win_length=self.n_fft,\n            window=window,\n            length=waveform.shape[-1],\n        )\n\n\ndef _load_wav_stdlib(path: str):\n    with wave.open(path, "rb") as wav:\n        channels = wav.getnchannels()\n        sample_width = wav.getsampwidth()\n        sample_rate = wav.getframerate()\n        frames = wav.readframes(wav.getnframes())\n\n    if sample_width == 2:\n        dtype = torch.int16\n        scale = float(2**15)\n    elif sample_width == 1:\n        dtype = torch.uint8\n        scale = 255.0\n    else:\n        raise ValueError(f"Unsupported WAV sample width: {sample_width} bytes")\n\n    data = torch.frombuffer(bytearray(frames), dtype=dtype).float()\n    if sample_width == 1:\n        data = data - 128.0\n    data = data.view(-1, channels).t().contiguous() / scale\n    return data, sample_rate\n\n\ndef _resample_scipy(waveform: torch.Tensor, orig_freq: int, new_freq: int) -> torch.Tensor:\n    try:\n        from scipy.signal import resample_poly\n    except ImportError as exc:\n        raise ImportError("Install torchaudio or scipy for audio resampling.") from exc\n\n    gcd = math.gcd(orig_freq, new_freq)\n    up = new_freq // gcd\n    down = orig_freq // gcd\n    resampled = resample_poly(waveform.numpy(), up=up, down=down, axis=-1)\n    return torch.from_numpy(resampled.copy()).float()\n\n\ndef _hz_to_mel(freq: torch.Tensor) -> torch.Tensor:\n    return 2595.0 * torch.log10(1.0 + freq / 700.0)\n\n\ndef _mel_to_hz(mels: torch.Tensor) -> torch.Tensor:\n    return 700.0 * (10.0 ** (mels / 2595.0) - 1.0)\n\n\ndef _mel_filterbank(sample_rate: int, n_fft: int, n_mels: int) -> torch.Tensor:\n    n_freqs = n_fft // 2 + 1\n    min_mel = _hz_to_mel(torch.tensor(0.0))\n    max_mel = _hz_to_mel(torch.tensor(float(sample_rate) / 2.0))\n    mels = torch.linspace(min_mel, max_mel, n_mels + 2)\n    hz = _mel_to_hz(mels)\n    bins = torch.floor((n_fft + 1) * hz / sample_rate).long().clamp(0, n_freqs - 1)\n\n    fb = torch.zeros(n_mels, n_freqs)\n    for i in range(n_mels):\n        left, center, right = bins[i].item(), bins[i + 1].item(), bins[i + 2].item()\n        if center > left:\n            fb[i, left:center] = torch.linspace(0.0, 1.0, center - left)\n        if right > center:\n            fb[i, center:right] = torch.linspace(1.0, 0.0, right - center)\n    return fb\n', 'coding/data.py': 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Optional\n\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\ntry:\n    from .config import DataConfig, LABEL_TO_ID\n    from .audio import AudioPipeline\nexcept ImportError:\n    from config import DataConfig, LABEL_TO_ID\n    from audio import AudioPipeline\n\n\nSPLITS = ("audio_visual_dataset_default", "audio_visual_dataset_robo_default")\n\n\ndef build_index(data_root: str | Path, skip_missing_files: bool = True) -> pd.DataFrame:\n    data_root = Path(data_root)\n    rows = []\n    skipped = 0\n\n    for split_name in SPLITS:\n        split_dir = data_root / split_name\n        csv_path = split_dir / "dataset.csv"\n        if not csv_path.exists():\n            continue\n\n        df = pd.read_csv(csv_path)\n        for item in df.to_dict("records"):\n            audio_path = split_dir / item["audio_file"]\n            image_path = split_dir / item["image_file"]\n            exists = audio_path.exists() and image_path.exists()\n            if skip_missing_files and not exists:\n                skipped += 1\n                continue\n            rows.append(\n                {\n                    "split_name": split_name,\n                    "audio_path": str(audio_path),\n                    "image_path": str(image_path),\n                    "label": item["category"],\n                    "label_id": LABEL_TO_ID[item["category"]],\n                    "files_exist": exists,\n                }\n            )\n\n    index = pd.DataFrame(rows)\n    index.attrs["skipped_missing_files"] = skipped\n    return index\n\n\ndef make_train_val_split(df: pd.DataFrame, val_size: float = 0.2, seed: int = 42):\n    try:\n        from sklearn.model_selection import train_test_split\n\n        train_df, val_df = train_test_split(\n            df,\n            test_size=val_size,\n            random_state=seed,\n            stratify=df["label_id"],\n        )\n    except Exception:\n        train_df = df.sample(frac=1.0 - val_size, random_state=seed)\n        val_df = df.drop(train_df.index)\n    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)\n\n\nclass AudioVisualDataset(Dataset):\n    def __init__(\n        self,\n        frame: pd.DataFrame,\n        config: DataConfig,\n        crop_mode: Optional[str] = None,\n        train: bool = False,\n    ) -> None:\n        self.frame = frame.reset_index(drop=True)\n        self.config = config\n        self.train = train\n        self.crop_mode = crop_mode or (config.train_crop if train else config.eval_crop)\n        self.audio = AudioPipeline(\n            target_sample_rate=config.target_sample_rate,\n            window_sec=config.audio_window_sec,\n            n_mels=config.n_mels,\n            n_fft=config.n_fft,\n            hop_length=config.hop_length,\n            normalize_db=config.normalize_audio_db,\n            spectral_gate=config.spectral_gate,\n            spectral_gate_noise_percentile=config.spectral_gate_noise_percentile,\n            spectral_gate_strength=config.spectral_gate_strength,\n        )\n        self.image_transform = transforms.Compose(\n            [\n                transforms.Resize((config.image_size, config.image_size)),\n                transforms.RandomHorizontalFlip(p=0.5 if train else 0.0),\n                transforms.ToTensor(),\n                transforms.Normalize(\n                    mean=(0.485, 0.456, 0.406),\n                    std=(0.229, 0.224, 0.225),\n                ),\n            ]\n        )\n\n    def __len__(self) -> int:\n        return len(self.frame)\n\n    def __getitem__(self, idx: int):\n        row = self.frame.iloc[idx]\n        waveform = self.audio.load_processed_waveform(row.audio_path, self.crop_mode)\n        mel = self.audio.waveform_to_mel(waveform)\n        image = Image.open(row.image_path).convert("RGB")\n        image = self.image_transform(image)\n        label = torch.tensor(row.label_id, dtype=torch.long)\n        binary_label = torch.tensor(0 if row.label == "ambient" else 1, dtype=torch.long)\n        return {\n            "audio": mel,\n            "waveform": waveform.squeeze(0),\n            "image": image,\n            "label": label,\n            "binary_label": binary_label,\n        }\n', 'coding/metrics.py': 'from __future__ import annotations\n\nimport numpy as np\nimport torch\n\ntry:\n    from .config import LABEL_TO_ID, LABELS\nexcept ImportError:\n    from config import LABEL_TO_ID, LABELS\n\n\ndef f1_scores(y_true, y_pred, num_classes: int = 4):\n    try:\n        from sklearn.metrics import f1_score\n\n        return {\n            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),\n            "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),\n        }\n    except Exception:\n        return _manual_f1_scores(y_true, y_pred, num_classes)\n\n\ndef paper_classification_metrics(y_true, y_pred, labels=None, paper_average: str = "weighted"):\n    labels = tuple(labels or LABELS)\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n    try:\n        from sklearn.metrics import (\n            accuracy_score,\n            confusion_matrix,\n            precision_recall_fscore_support,\n        )\n\n        macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(\n            y_true, y_pred, average="macro", zero_division=0\n        )\n        weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(\n            y_true, y_pred, average="weighted", zero_division=0\n        )\n        paper_p, paper_r, paper_f1, _ = precision_recall_fscore_support(\n            y_true, y_pred, average=paper_average, zero_division=0\n        )\n        per_p, per_r, per_f1, support = precision_recall_fscore_support(\n            y_true, y_pred, labels=list(range(len(labels))), zero_division=0\n        )\n        return {\n            "paper_average": paper_average,\n            "paper_f1": float(paper_f1),\n            "paper_precision": float(paper_p),\n            "paper_recall": float(paper_r),\n            "macro_f1": float(macro_f1),\n            "macro_precision": float(macro_p),\n            "macro_recall": float(macro_r),\n            "weighted_f1": float(weighted_f1),\n            "weighted_precision": float(weighted_p),\n            "weighted_recall": float(weighted_r),\n            "accuracy": float(accuracy_score(y_true, y_pred)),\n            "per_class": {\n                label: {\n                    "precision": float(per_p[idx]),\n                    "recall": float(per_r[idx]),\n                    "f1": float(per_f1[idx]),\n                    "support": int(support[idx]),\n                }\n                for idx, label in enumerate(labels)\n            },\n            "confusion_matrix": confusion_matrix(\n                y_true, y_pred, labels=list(range(len(labels)))\n            ).tolist(),\n        }\n    except Exception:\n        out = _manual_f1_scores(y_true, y_pred, len(labels))\n        out.update(\n            {\n                "paper_average": paper_average,\n                "paper_f1": out["weighted_f1"] if paper_average == "weighted" else out["macro_f1"],\n                "accuracy": float((y_true == y_pred).mean()),\n            }\n        )\n        return out\n\n\ndef binary_contact_f1(y_true, y_pred):\n    ambient_id = LABEL_TO_ID["ambient"]\n    true_contact = np.asarray(y_true) != ambient_id\n    pred_contact = np.asarray(y_pred) != ambient_id\n    try:\n        from sklearn.metrics import f1_score\n\n        return float(f1_score(true_contact, pred_contact, zero_division=0))\n    except Exception:\n        tp = float(np.logical_and(true_contact, pred_contact).sum())\n        fp = float(np.logical_and(~true_contact, pred_contact).sum())\n        fn = float(np.logical_and(true_contact, ~pred_contact).sum())\n        denom = (2 * tp + fp + fn)\n        return 0.0 if denom == 0 else (2 * tp / denom)\n\n\ndef binary_contact_metrics(y_true, y_pred, paper_average: str = "binary"):\n    ambient_id = LABEL_TO_ID["ambient"]\n    true_contact = np.asarray(y_true) != ambient_id\n    pred_contact = np.asarray(y_pred) != ambient_id\n    return binary_metrics_from_ids(true_contact.astype(int), pred_contact.astype(int), paper_average)\n\n\ndef binary_metrics_from_ids(y_true, y_pred, paper_average: str = "binary"):\n    y_true = np.asarray(y_true).astype(int)\n    y_pred = np.asarray(y_pred).astype(int)\n    try:\n        from sklearn.metrics import (\n            accuracy_score,\n            confusion_matrix,\n            precision_recall_fscore_support,\n        )\n\n        average = "binary" if paper_average == "binary" else paper_average\n        p, r, f1, _ = precision_recall_fscore_support(\n            y_true, y_pred, average=average, pos_label=1, zero_division=0\n        )\n        per_p, per_r, per_f1, support = precision_recall_fscore_support(\n            y_true, y_pred, labels=[0, 1], zero_division=0\n        )\n        return {\n            "paper_average": paper_average,\n            "paper_f1": float(f1),\n            "paper_precision": float(p),\n            "paper_recall": float(r),\n            "binary_contact_f1": float(f1),\n            "accuracy": float(accuracy_score(y_true, y_pred)),\n            "per_class": {\n                "non_contact": {\n                    "precision": float(per_p[0]),\n                    "recall": float(per_r[0]),\n                    "f1": float(per_f1[0]),\n                    "support": int(support[0]),\n                },\n                "contact": {\n                    "precision": float(per_p[1]),\n                    "recall": float(per_r[1]),\n                    "f1": float(per_f1[1]),\n                    "support": int(support[1]),\n                },\n            },\n            "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),\n        }\n    except Exception:\n        f1 = _binary_f1_np(y_true, y_pred)\n        return {\n            "paper_average": paper_average,\n            "paper_f1": f1,\n            "binary_contact_f1": f1,\n            "accuracy": float((y_true == y_pred).mean()),\n        }\n\n\ndef _binary_f1_np(y_true, y_pred):\n    y_true = np.asarray(y_true).astype(bool)\n    y_pred = np.asarray(y_pred).astype(bool)\n    tp = float(np.logical_and(y_true, y_pred).sum())\n    fp = float(np.logical_and(~y_true, y_pred).sum())\n    fn = float(np.logical_and(y_true, ~y_pred).sum())\n    denom = (2 * tp + fp + fn)\n    return 0.0 if denom == 0 else float(2 * tp / denom)\n\n\ndef collect_predictions(logits_list, labels_list):\n    logits = torch.cat(logits_list, dim=0)\n    labels = torch.cat(labels_list, dim=0)\n    preds = logits.argmax(dim=1)\n    return labels.cpu().numpy(), preds.cpu().numpy()\n\n\ndef _manual_f1_scores(y_true, y_pred, num_classes: int):\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n    per_class = []\n    supports = []\n    for cls in range(num_classes):\n        true_cls = y_true == cls\n        pred_cls = y_pred == cls\n        tp = float(np.logical_and(true_cls, pred_cls).sum())\n        fp = float(np.logical_and(~true_cls, pred_cls).sum())\n        fn = float(np.logical_and(true_cls, ~pred_cls).sum())\n        support = float(true_cls.sum())\n        denom = 2 * tp + fp + fn\n        per_class.append(0.0 if denom == 0 else 2 * tp / denom)\n        supports.append(support)\n    supports = np.asarray(supports)\n    per_class = np.asarray(per_class)\n    weighted = 0.0 if supports.sum() == 0 else float((per_class * supports).sum() / supports.sum())\n    return {"macro_f1": float(per_class.mean()), "weighted_f1": weighted}\n', 'coding/paper_model.py': 'from __future__ import annotations\n\nimport torch\nfrom torch import nn\nfrom torchvision import models\n\n\nclass PaperLikeFusionNet(nn.Module):\n    """AST + CLAP + ViT-B/16 with lightweight Transformer fusion."""\n\n    def __init__(\n        self,\n        num_classes: int = 4,\n        sample_rate: int = 22000,\n        ast_model_name: str = "MIT/ast-finetuned-audioset-10-10-0.4593",\n        clap_model_name: str = "laion/clap-htsat-unfused",\n        fusion_dim: int = 256,\n        fusion_heads: int = 4,\n        fusion_layers: int = 2,\n        fusion_dropout: float = 0.1,\n        freeze_pretrained: bool = False,\n        ast_input_source: str = "mel",\n    ) -> None:\n        super().__init__()\n        try:\n            from transformers import ASTFeatureExtractor, ASTModel, AutoProcessor, ClapModel\n        except ImportError as exc:\n            raise ImportError(\n                "PaperLikeFusionNet requires transformers. On Kaggle, install/enable "\n                "`transformers` and allow model weights from the Kaggle cache/input or internet."\n            ) from exc\n\n        self.sample_rate = sample_rate\n        self.ast_input_source = ast_input_source\n        self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained(ast_model_name)\n        self.ast_model = ASTModel.from_pretrained(ast_model_name)\n        self.clap_processor = AutoProcessor.from_pretrained(clap_model_name)\n        self.clap_model = ClapModel.from_pretrained(clap_model_name)\n        self.ast_sample_rate = getattr(self.ast_feature_extractor, "sampling_rate", sample_rate)\n        self.clap_sample_rate = getattr(\n            getattr(self.clap_processor, "feature_extractor", None),\n            "sampling_rate",\n            sample_rate,\n        )\n\n        vit_weights = models.ViT_B_16_Weights.DEFAULT\n        self.image_model = models.vit_b_16(weights=vit_weights)\n        vit_dim = self.image_model.heads.head.in_features\n        self.image_model.heads = nn.Identity()\n\n        ast_dim = self.ast_model.config.hidden_size\n        clap_dim = self.clap_model.config.projection_dim\n        self.ast_proj = nn.Linear(ast_dim, fusion_dim)\n        self.clap_proj = nn.Linear(clap_dim, fusion_dim)\n        self.image_proj = nn.Linear(vit_dim, fusion_dim)\n\n        self.cls_token = nn.Parameter(torch.zeros(1, 1, fusion_dim))\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=fusion_dim,\n            nhead=fusion_heads,\n            dim_feedforward=fusion_dim * 4,\n            dropout=fusion_dropout,\n            activation="gelu",\n            batch_first=True,\n            norm_first=False,\n        )\n        self.fusion = nn.TransformerEncoder(encoder_layer, num_layers=fusion_layers)\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(fusion_dim),\n            nn.Linear(fusion_dim, fusion_dim),\n            nn.GELU(),\n            nn.Dropout(fusion_dropout),\n            nn.Linear(fusion_dim, num_classes),\n        )\n\n        if freeze_pretrained:\n            self._freeze_pretrained()\n\n    def _freeze_pretrained(self) -> None:\n        for module in (self.ast_model, self.clap_model, self.image_model):\n            for param in module.parameters():\n                param.requires_grad = False\n\n    def forward(\n        self,\n        waveform: torch.Tensor,\n        image: torch.Tensor,\n        audio: torch.Tensor | None = None,\n    ) -> torch.Tensor:\n        if self.ast_input_source == "mel" and audio is not None:\n            ast_emb = self.encode_ast_mel(audio)\n        else:\n            ast_emb = self.encode_ast(waveform)\n        clap_emb = self.encode_clap(waveform)\n        image_emb = self.image_model(image)\n\n        tokens = torch.stack(\n            [\n                self.ast_proj(ast_emb),\n                self.clap_proj(clap_emb),\n                self.image_proj(image_emb),\n            ],\n            dim=1,\n        )\n        cls = self.cls_token.expand(tokens.size(0), -1, -1)\n        fused = self.fusion(torch.cat([cls, tokens], dim=1))\n        return self.classifier(fused[:, 0])\n\n    def encode_ast(self, waveform: torch.Tensor) -> torch.Tensor:\n        device = waveform.device\n        arrays = self._to_processor_arrays(waveform, self.ast_sample_rate)\n        inputs = self.ast_feature_extractor(\n            arrays,\n            sampling_rate=self.ast_sample_rate,\n            return_tensors="pt",\n            padding=True,\n        )\n        inputs = {key: value.to(device) for key, value in inputs.items()}\n        outputs = self.ast_model(**inputs)\n        return _extract_model_embedding(outputs)\n\n    def encode_ast_mel(self, mel: torch.Tensor) -> torch.Tensor:\n        """Feed the explicit mel-spectrogram from AudioPipeline into AST.\n\n        AudioPipeline returns normalized dB mel as [B, 1, mel_bins, frames].\n        AST expects normalized log-mel frames as [B, frames, mel_bins].\n        """\n        if mel.ndim == 3:\n            mel = mel.unsqueeze(1)\n        mel_db = mel.squeeze(1)\n        if mel_db.min() >= 0.0 and mel_db.max() <= 1.0:\n            mel_db = mel_db * 80.0 - 80.0\n        input_values = mel_db.transpose(1, 2)\n        input_values = self._fit_ast_frames(input_values)\n        mean = float(getattr(self.ast_feature_extractor, "mean", 0.0))\n        std = float(getattr(self.ast_feature_extractor, "std", 1.0))\n        input_values = (input_values - mean) / max(std, 1e-8)\n        outputs = self.ast_model(input_values=input_values.to(mel.device))\n        return _extract_model_embedding(outputs)\n\n    def _fit_ast_frames(self, input_values: torch.Tensor) -> torch.Tensor:\n        max_length = int(getattr(self.ast_feature_extractor, "max_length", input_values.shape[1]))\n        if input_values.shape[1] > max_length:\n            return input_values[:, :max_length, :]\n        if input_values.shape[1] == max_length:\n            return input_values\n        pad = input_values.new_zeros(\n            input_values.shape[0],\n            max_length - input_values.shape[1],\n            input_values.shape[2],\n        )\n        return torch.cat([input_values, pad], dim=1)\n\n    def encode_clap(self, waveform: torch.Tensor) -> torch.Tensor:\n        device = waveform.device\n        arrays = self._to_processor_arrays(waveform, self.clap_sample_rate)\n        inputs = self.clap_processor(\n            audio=arrays,\n            sampling_rate=self.clap_sample_rate,\n            return_tensors="pt",\n            padding=True,\n        )\n        inputs = {key: value.to(device) for key, value in inputs.items()}\n        return _extract_model_embedding(self.clap_model.get_audio_features(**inputs))\n\n    def _to_processor_arrays(self, waveform: torch.Tensor, target_rate: int):\n        arrays = [item.detach().float().cpu().numpy() for item in waveform]\n        if target_rate == self.sample_rate:\n            return arrays\n        try:\n            from scipy.signal import resample_poly\n        except ImportError as exc:\n            raise ImportError("scipy is required to adapt waveform sample rates for pretrained processors.") from exc\n\n        import math\n\n        gcd = math.gcd(self.sample_rate, target_rate)\n        up = target_rate // gcd\n        down = self.sample_rate // gcd\n        return [resample_poly(item, up=up, down=down).astype("float32") for item in arrays]\n\n\ndef _extract_model_embedding(output):\n    if torch.is_tensor(output):\n        return output\n\n    pooler = getattr(output, "pooler_output", None)\n    if torch.is_tensor(pooler):\n        return pooler\n\n    last_hidden = getattr(output, "last_hidden_state", None)\n    if torch.is_tensor(last_hidden):\n        return last_hidden[:, 0]\n\n    if isinstance(output, (tuple, list)):\n        for item in output:\n            if torch.is_tensor(item):\n                return item[:, 0] if item.ndim == 3 else item\n            nested = _extract_model_embedding(item)\n            if torch.is_tensor(nested):\n                return nested\n\n    raise TypeError(f"Could not extract tensor embedding from output type: {type(output)!r}")\n\n\nclass ASTCLAPAudioNet(PaperLikeFusionNet):\n    """Audio-only AST + CLAP branch with the same fusion head style."""\n\n    def __init__(\n        self,\n        num_classes: int = 4,\n        sample_rate: int = 16000,\n        ast_model_name: str = "MIT/ast-finetuned-audioset-10-10-0.4593",\n        clap_model_name: str = "laion/clap-htsat-unfused",\n        fusion_dim: int = 256,\n        fusion_heads: int = 4,\n        fusion_layers: int = 2,\n        fusion_dropout: float = 0.1,\n        freeze_pretrained: bool = False,\n        ast_input_source: str = "mel",\n    ) -> None:\n        nn.Module.__init__(self)\n        try:\n            from transformers import ASTFeatureExtractor, ASTModel, AutoProcessor, ClapModel\n        except ImportError as exc:\n            raise ImportError("ASTCLAPAudioNet requires transformers.") from exc\n\n        self.sample_rate = sample_rate\n        self.ast_input_source = ast_input_source\n        self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained(ast_model_name)\n        self.ast_model = ASTModel.from_pretrained(ast_model_name)\n        self.clap_processor = AutoProcessor.from_pretrained(clap_model_name)\n        self.clap_model = ClapModel.from_pretrained(clap_model_name)\n        self.ast_sample_rate = getattr(self.ast_feature_extractor, "sampling_rate", sample_rate)\n        self.clap_sample_rate = getattr(\n            getattr(self.clap_processor, "feature_extractor", None),\n            "sampling_rate",\n            sample_rate,\n        )\n\n        ast_dim = self.ast_model.config.hidden_size\n        clap_dim = self.clap_model.config.projection_dim\n        self.ast_proj = nn.Linear(ast_dim, fusion_dim)\n        self.clap_proj = nn.Linear(clap_dim, fusion_dim)\n        self.cls_token = nn.Parameter(torch.zeros(1, 1, fusion_dim))\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=fusion_dim,\n            nhead=fusion_heads,\n            dim_feedforward=fusion_dim * 4,\n            dropout=fusion_dropout,\n            activation="gelu",\n            batch_first=True,\n            norm_first=False,\n        )\n        self.fusion = nn.TransformerEncoder(encoder_layer, num_layers=fusion_layers)\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(fusion_dim),\n            nn.Linear(fusion_dim, fusion_dim),\n            nn.GELU(),\n            nn.Dropout(fusion_dropout),\n            nn.Linear(fusion_dim, num_classes),\n        )\n\n        if freeze_pretrained:\n            for module in (self.ast_model, self.clap_model):\n                for param in module.parameters():\n                    param.requires_grad = False\n\n    def forward(\n        self,\n        waveform: torch.Tensor,\n        image: torch.Tensor | None = None,\n        audio: torch.Tensor | None = None,\n    ) -> torch.Tensor:\n        if self.ast_input_source == "mel" and audio is not None:\n            ast_emb = self.encode_ast_mel(audio)\n        else:\n            ast_emb = self.encode_ast(waveform)\n        clap_emb = self.encode_clap(waveform)\n        tokens = torch.stack([self.ast_proj(ast_emb), self.clap_proj(clap_emb)], dim=1)\n        cls = self.cls_token.expand(tokens.size(0), -1, -1)\n        fused = self.fusion(torch.cat([cls, tokens], dim=1))\n        return self.classifier(fused[:, 0])\n\n\nclass ViTImageNet(nn.Module):\n    """Image-only ViT-B/16 branch for the released single-frame dataset."""\n\n    def __init__(\n        self,\n        num_classes: int = 4,\n        fusion_dim: int = 256,\n        fusion_dropout: float = 0.1,\n        freeze_pretrained: bool = False,\n    ) -> None:\n        super().__init__()\n        self.image_model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)\n        vit_dim = self.image_model.heads.head.in_features\n        self.image_model.heads = nn.Identity()\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(vit_dim),\n            nn.Linear(vit_dim, fusion_dim),\n            nn.GELU(),\n            nn.Dropout(fusion_dropout),\n            nn.Linear(fusion_dim, num_classes),\n        )\n        if freeze_pretrained:\n            for param in self.image_model.parameters():\n                param.requires_grad = False\n\n    def forward(\n        self,\n        waveform: torch.Tensor | None = None,\n        image: torch.Tensor | None = None,\n        audio: torch.Tensor | None = None,\n    ) -> torch.Tensor:\n        return self.classifier(self.image_model(image))\n', 'coding/train_paper.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader\nfrom tqdm.auto import tqdm\n\ntry:\n    from .config import DataConfig, LABELS, ModelConfig, TrainConfig\n    from .data import AudioVisualDataset, build_index, make_train_val_split\n    from .metrics import binary_contact_metrics, collect_predictions, paper_classification_metrics\n    from .paper_model import ASTCLAPAudioNet, PaperLikeFusionNet, ViTImageNet\nexcept ImportError:\n    from config import DataConfig, LABELS, ModelConfig, TrainConfig\n    from data import AudioVisualDataset, build_index, make_train_val_split\n    from metrics import binary_contact_metrics, collect_predictions, paper_classification_metrics\n    from paper_model import ASTCLAPAudioNet, PaperLikeFusionNet, ViTImageNet\n\n\nTASK_LABELS = {\n    "multiclass": LABELS,\n    "binary": ("non_contact", "contact"),\n}\n\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef run_epoch(\n    model,\n    loader,\n    criterion,\n    optimizer=None,\n    device="cpu",\n    use_amp=True,\n    task="multiclass",\n    paper_average="weighted",\n):\n    is_train = optimizer is not None\n    model.train(is_train)\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and device == "cuda")\n    total_loss = 0.0\n    total_items = 0\n    logits_list = []\n    labels_list = []\n\n    for batch in tqdm(loader, leave=False):\n        waveform = batch["waveform"].to(device)\n        audio = batch["audio"].to(device)\n        image = batch["image"].to(device)\n        label_key = "binary_label" if task == "binary" else "label"\n        labels = batch[label_key].to(device)\n\n        with torch.set_grad_enabled(is_train):\n            with torch.amp.autocast("cuda", enabled=use_amp and device == "cuda"):\n                logits = model(waveform=waveform, image=image, audio=audio)\n                loss = criterion(logits, labels)\n\n            if is_train:\n                optimizer.zero_grad(set_to_none=True)\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n\n        total_loss += loss.item() * labels.size(0)\n        total_items += labels.size(0)\n        logits_list.append(logits.detach().cpu())\n        labels_list.append(labels.detach().cpu())\n\n    y_true, y_pred = collect_predictions(logits_list, labels_list)\n    if task == "binary":\n        metrics = binary_contact_metrics(y_true, y_pred, paper_average="binary")\n        metrics["macro_f1"] = paper_classification_metrics(\n            y_true,\n            y_pred,\n            labels=TASK_LABELS["binary"],\n            paper_average="macro",\n        )["macro_f1"]\n        metrics["weighted_f1"] = paper_classification_metrics(\n            y_true,\n            y_pred,\n            labels=TASK_LABELS["binary"],\n            paper_average="weighted",\n        )["weighted_f1"]\n    else:\n        metrics = paper_classification_metrics(\n            y_true,\n            y_pred,\n            labels=LABELS,\n            paper_average=paper_average,\n        )\n        binary = binary_contact_metrics(y_true, y_pred, paper_average="binary")\n        metrics["binary_contact_f1"] = binary["binary_contact_f1"]\n        metrics["binary_contact"] = binary\n    metrics["loss"] = total_loss / total_items\n    return metrics\n\n\ndef make_class_weight_tensor(train_df, device: str, task: str):\n    if task == "binary":\n        ids = (train_df["label"] != "ambient").astype(int)\n        counts = ids.value_counts().reindex(range(2)).fillna(0).to_numpy(dtype=np.float32)\n        labels = TASK_LABELS["binary"]\n    else:\n        counts = train_df["label_id"].value_counts().reindex(range(len(LABELS))).fillna(0).to_numpy(dtype=np.float32)\n        labels = LABELS\n    weights = counts.sum() / np.maximum(counts, 1.0)\n    weights = weights / weights.mean()\n    print(f"Class weights: {dict(zip(labels, weights.round(4).tolist()))}")\n    return torch.tensor(weights, dtype=torch.float32, device=device)\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-root", type=Path, default=Path("dataset"))\n    parser.add_argument("--epochs", type=int, default=5)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--num-workers", type=int, default=2)\n    parser.add_argument("--target-sample-rate", type=int, default=16000)\n    parser.add_argument("--audio-window-sec", type=float, default=0.8)\n    parser.add_argument("--train-crop", choices=("random", "center", "energy"), default="random")\n    parser.add_argument("--eval-crop", choices=("center", "energy"), default="center")\n    parser.add_argument("--ast-model-name", default=ModelConfig.ast_model_name)\n    parser.add_argument("--clap-model-name", default=ModelConfig.clap_model_name)\n    parser.add_argument("--fusion-dim", type=int, default=256)\n    parser.add_argument("--fusion-layers", type=int, default=2)\n    parser.add_argument("--fusion-heads", type=int, default=4)\n    parser.add_argument("--freeze-pretrained", action="store_true")\n    parser.add_argument("--no-class-weights", action="store_true")\n    parser.add_argument("--task", choices=("multiclass", "binary"), default="multiclass")\n    parser.add_argument("--mode", choices=("audio", "video", "fusion"), default="fusion")\n    parser.add_argument("--split-strategy", choices=("domain", "random"), default="domain")\n    parser.add_argument("--paper-average", choices=("weighted", "macro", "micro"), default="weighted")\n    parser.add_argument("--ast-input-source", choices=("mel", "waveform"), default="mel")\n    parser.add_argument("--output-dir", type=Path, default=Path("outputs"))\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    train_cfg = TrainConfig(\n        epochs=args.epochs,\n        batch_size=args.batch_size,\n        lr=args.lr,\n        num_workers=args.num_workers,\n        class_weights=not args.no_class_weights,\n    )\n    data_cfg = DataConfig(\n        data_root=args.data_root,\n        target_sample_rate=args.target_sample_rate,\n        audio_window_sec=args.audio_window_sec,\n        train_crop=args.train_crop,\n        eval_crop=args.eval_crop,\n    )\n    model_cfg = ModelConfig(\n        ast_model_name=args.ast_model_name,\n        clap_model_name=args.clap_model_name,\n        fusion_dim=args.fusion_dim,\n        fusion_layers=args.fusion_layers,\n        fusion_heads=args.fusion_heads,\n        freeze_pretrained=args.freeze_pretrained,\n        ast_input_source=args.ast_input_source,\n    )\n    set_seed(train_cfg.seed)\n\n    index = build_index(data_cfg.data_root, skip_missing_files=data_cfg.skip_missing_files)\n    print(f"Indexed {len(index)} samples. Skipped missing files: {index.attrs.get(\'skipped_missing_files\', 0)}")\n    print(index["label"].value_counts().reindex(LABELS).fillna(0).astype(int))\n\n    if args.split_strategy == "domain":\n        train_df = index[index["split_name"] == "audio_visual_dataset_default"].reset_index(drop=True)\n        val_df = index[index["split_name"] == "audio_visual_dataset_robo_default"].reset_index(drop=True)\n    else:\n        train_df, val_df = make_train_val_split(index, train_cfg.val_size, train_cfg.seed)\n    print(f"Split strategy: {args.split_strategy}")\n    print(f"Task: {args.task}")\n    print(f"Mode: {args.mode}")\n    train_ds = AudioVisualDataset(train_df, data_cfg, train=True)\n    val_ds = AudioVisualDataset(val_df, data_cfg, train=False)\n    train_loader = DataLoader(\n        train_ds,\n        batch_size=train_cfg.batch_size,\n        shuffle=True,\n        num_workers=train_cfg.num_workers,\n        pin_memory=True,\n    )\n    val_loader = DataLoader(\n        val_ds,\n        batch_size=train_cfg.batch_size,\n        shuffle=False,\n        num_workers=train_cfg.num_workers,\n        pin_memory=True,\n    )\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    if args.mode == "audio":\n        model = ASTCLAPAudioNet(\n            num_classes=len(TASK_LABELS[args.task]),\n            sample_rate=data_cfg.target_sample_rate,\n            ast_model_name=model_cfg.ast_model_name,\n            clap_model_name=model_cfg.clap_model_name,\n            fusion_dim=model_cfg.fusion_dim,\n            fusion_heads=model_cfg.fusion_heads,\n            fusion_layers=model_cfg.fusion_layers,\n            fusion_dropout=model_cfg.fusion_dropout,\n            freeze_pretrained=model_cfg.freeze_pretrained,\n            ast_input_source=model_cfg.ast_input_source,\n        )\n    elif args.mode == "video":\n        model = ViTImageNet(\n            num_classes=len(TASK_LABELS[args.task]),\n            fusion_dim=model_cfg.fusion_dim,\n            fusion_dropout=model_cfg.fusion_dropout,\n            freeze_pretrained=model_cfg.freeze_pretrained,\n        )\n    else:\n        model = PaperLikeFusionNet(\n            num_classes=len(TASK_LABELS[args.task]),\n            sample_rate=data_cfg.target_sample_rate,\n            ast_model_name=model_cfg.ast_model_name,\n            clap_model_name=model_cfg.clap_model_name,\n            fusion_dim=model_cfg.fusion_dim,\n            fusion_heads=model_cfg.fusion_heads,\n            fusion_layers=model_cfg.fusion_layers,\n            fusion_dropout=model_cfg.fusion_dropout,\n            freeze_pretrained=model_cfg.freeze_pretrained,\n            ast_input_source=model_cfg.ast_input_source,\n        )\n    model = model.to(device)\n    criterion = nn.CrossEntropyLoss(\n        weight=make_class_weight_tensor(train_df, device, args.task) if train_cfg.class_weights else None\n    )\n    optimizer = torch.optim.AdamW(\n        [param for param in model.parameters() if param.requires_grad],\n        lr=train_cfg.lr,\n        weight_decay=train_cfg.weight_decay,\n    )\n\n    out_dir = args.output_dir\n    out_dir.mkdir(parents=True, exist_ok=True)\n    best_paper_f1 = -1.0\n    best_metrics = None\n    history = []\n\n    for epoch in range(1, train_cfg.epochs + 1):\n        train_metrics = run_epoch(\n            model,\n            train_loader,\n            criterion,\n            optimizer,\n            device,\n            train_cfg.use_amp,\n            args.task,\n            args.paper_average,\n        )\n        val_metrics = run_epoch(\n            model,\n            val_loader,\n            criterion,\n            None,\n            device,\n            train_cfg.use_amp,\n            args.task,\n            args.paper_average,\n        )\n        history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})\n        print(\n            f"epoch={epoch} "\n            f"train_loss={train_metrics[\'loss\']:.4f} train_macro_f1={train_metrics[\'macro_f1\']:.4f} "\n            f"train_paper_f1={train_metrics[\'paper_f1\']:.4f} "\n            f"val_loss={val_metrics[\'loss\']:.4f} val_macro_f1={val_metrics[\'macro_f1\']:.4f} "\n            f"val_weighted_f1={val_metrics[\'weighted_f1\']:.4f} "\n            f"val_paper_f1={val_metrics[\'paper_f1\']:.4f} "\n            f"val_precision={val_metrics[\'paper_precision\']:.4f} val_recall={val_metrics[\'paper_recall\']:.4f}"\n        )\n        if args.task == "multiclass":\n            print(f"val_binary_f1_from_multiclass={val_metrics[\'binary_contact_f1\']:.4f}")\n        if val_metrics["paper_f1"] > best_paper_f1:\n            best_paper_f1 = val_metrics["paper_f1"]\n            best_metrics = val_metrics\n            torch.save(\n                {\n                    "model": model.state_dict(),\n                    "labels": TASK_LABELS[args.task],\n                    "mode": args.mode,\n                    "task": args.task,\n                    "split_strategy": args.split_strategy,\n                    "paper_average": args.paper_average,\n                    "data_config": data_cfg.__dict__,\n                    "model_config": model_cfg.__dict__,\n                    "val_metrics": val_metrics,\n                },\n                out_dir / f"best_{args.task}_{args.mode}_paper_like_model.pt",\n            )\n\n    print(f"Best validation paper-style F1: {best_paper_f1:.4f}")\n    result = {\n        "mode": args.mode,\n        "task": args.task,\n        "split_strategy": args.split_strategy,\n        "paper_average": args.paper_average if args.task == "multiclass" else "binary",\n        "best_paper_f1": best_paper_f1,\n        "best_val_metrics": best_metrics,\n        "weight_file": str(out_dir / f"best_{args.task}_{args.mode}_paper_like_model.pt"),\n        "history": history,\n    }\n    result_path = out_dir / f"{args.task}_{args.mode}_paper_results.json"\n    result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")\n    print(f"Saved result: {result_path}")\n    if best_metrics is not None:\n        print(\n            "BEST "\n            f"mode={args.mode} task={args.task} "\n            f"F1={best_metrics.get(\'paper_f1\', float(\'nan\')):.4f} "\n            f"Precision={best_metrics.get(\'paper_precision\', float(\'nan\')):.4f} "\n            f"Recall={best_metrics.get(\'paper_recall\', float(\'nan\')):.4f} "\n            f"macro_f1={best_metrics.get(\'macro_f1\', float(\'nan\')):.4f} "\n            f"weighted_f1={best_metrics.get(\'weighted_f1\', float(\'nan\')):.4f} "\n            f"acc={best_metrics.get(\'accuracy\', float(\'nan\')):.4f}"\n        )\n\n\nif __name__ == "__main__":\n    main()\n'}

def find_project_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working'), Path('/kaggle/input')]:
        if (candidate / 'coding' / 'train_paper.py').exists():
            return candidate
        if candidate.exists():
            for child in candidate.glob('*'):
                if (child / 'coding' / 'train_paper.py').exists():
                    return child
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('/kaggle/working/codex_bootstrap') if Path('/kaggle/working').exists() else Path('codex_bootstrap')
    for rel_path, content in EMBEDDED_FILES.items():
        path = PROJECT_ROOT / rel_path
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding='utf-8')
    print('Bootstrapped embedded coding/ files to', PROJECT_ROOT.resolve())

PROJECT_ROOT = Path(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT.resolve())
print('DATA_ROOT =', DATA_ROOT.resolve() if DATA_ROOT.exists() else DATA_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR.resolve())
print('train_paper.py exists =', (PROJECT_ROOT / 'coding' / 'train_paper.py').exists())
print('dataset exists =', DATA_ROOT.exists())


If Kaggle does not include `transformers`, uncomment the install cell. Pretrained AST/CLAP/ViT weights require Kaggle internet or an attached cache/model dataset.

In [ ]:
# !pip install -q transformers

## 2. Imports and Config

In [ ]:
# Paper-style controls. Edit these before running training cells.
EPOCHS = 5
BATCH_SIZE = 4
LR = 1e-4
NUM_WORKERS = 2
SPLIT_STRATEGY = 'domain'  # domain = train default/hand-held, validate robo/robot
PAPER_AVERAGE = 'weighted' # paper does not specify; weighted matches their table most closely
AST_INPUT_SOURCE = 'mel'   # explicit mel-spectrogram into AST; CLAP still uses its processor
FREEZE_PRETRAINED = True
CLASS_WEIGHTS = True

RUN_MODES = ['audio', 'video', 'fusion']
RUN_TASKS = ['multiclass', 'binary']

print('RUN_MODES =', RUN_MODES)
print('RUN_TASKS =', RUN_TASKS)
print('FREEZE_PRETRAINED =', FREEZE_PRETRAINED)


## 3. Dataset Check


In [ ]:
from coding.config import LABELS
from coding.data import build_index

index = build_index(DATA_ROOT, skip_missing_files=True)
print('Valid samples:', len(index))
print('Skipped missing files:', index.attrs.get('skipped_missing_files', 0))
display(index['split_name'].value_counts())
display(index['label'].value_counts().reindex(LABELS).fillna(0).astype(int))

train_df = index[index['split_name'] == 'audio_visual_dataset_default'].reset_index(drop=True)
val_df = index[index['split_name'] == 'audio_visual_dataset_robo_default'].reset_index(drop=True)
print('Train:', len(train_df), 'Val:', len(val_df))
print('Train labels:')
display(train_df['label'].value_counts().reindex(LABELS).fillna(0).astype(int))
print('Val labels:')
display(val_df['label'].value_counts().reindex(LABELS).fillna(0).astype(int))


## 4. Training Helpers


In [ ]:
def run_train(mode, task):
    cmd = [
        sys.executable, str(PROJECT_ROOT / 'coding' / 'train_paper.py'),
        '--data-root', str(DATA_ROOT),
        '--output-dir', str(OUTPUT_DIR),
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--lr', str(LR),
        '--num-workers', str(NUM_WORKERS),
        '--split-strategy', SPLIT_STRATEGY,
        '--paper-average', PAPER_AVERAGE,
        '--ast-input-source', AST_INPUT_SOURCE,
        '--mode', mode,
        '--task', task,
    ]
    if FREEZE_PRETRAINED:
        cmd.append('--freeze-pretrained')
    if not CLASS_WEIGHTS:
        cmd.append('--no-class-weights')
    print('
' + '=' * 90)
    print('RUN:', ' '.join(cmd))
    print('=' * 90)
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print('STDERR:')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Training failed for mode={mode}, task={task}')
    result_path = OUTPUT_DIR / f'{task}_{mode}_paper_results.json'
    with open(result_path, 'r', encoding='utf-8') as f:
        payload = json.load(f)
    return payload

def result_row(result):
    m = result['best_val_metrics']
    return {
        'task': result['task'],
        'mode': result['mode'],
        'F1': m.get('paper_f1'),
        'Precision': m.get('paper_precision'),
        'Recall': m.get('paper_recall'),
        'macro_f1': m.get('macro_f1'),
        'weighted_f1': m.get('weighted_f1'),
        'accuracy': m.get('accuracy'),
        'binary_contact_f1': m.get('binary_contact_f1'),
        'weight_file': result.get('weight_file'),
    }

def show_results(results):
    rows = [result_row(r) for r in results]
    df = pd.DataFrame(rows)
    display(df)
    return df


## 5. Train Multiclass


In [ ]:
multiclass_results = []
for mode in RUN_MODES:
    multiclass_results.append(run_train(mode, 'multiclass'))
multiclass_summary = show_results(multiclass_results)


## 6. Train Binary Contact


In [ ]:
binary_results = []
for mode in RUN_MODES:
    binary_results.append(run_train(mode, 'binary'))
binary_summary = show_results(binary_results)


## 7. Combined Summary


In [ ]:
all_results = []
if 'multiclass_results' in globals():
    all_results.extend(multiclass_results)
if 'binary_results' in globals():
    all_results.extend(binary_results)
combined_summary = show_results(all_results)
combined_summary.to_csv(OUTPUT_DIR / 'paper_style_summary.csv', index=False)
print('Saved summary:', OUTPUT_DIR / 'paper_style_summary.csv')


## 8. Saved Outputs


In [ ]:
print('Output files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)
